# 🌊 WaterMamba: Underwater Image Enhancement on Kaggle (GPU)
**Paper**: arXiv:2405.08419 — *WaterMamba: Visual State Space Model for Underwater Image Enhancement*

### ⚙️ Kaggle Settings:
1. **Accelerator**: Select **GPU P100** or **GPU T4 x2** (Settings panel on right 👉).
2. **Internet**: Set to **ON** (needed for `pip install mamba-ssm`).
3. **Persistence**: Variables and files in `/kaggle/working/` can be downloaded.

In [ ]:
# Step 1: Install official fused CUDA SSM kernels for Linux
# Use ninja and --no-build-isolation so pip can access Kaggle's pre-installed CUDA/PyTorch
!pip install -q ninja packaging
!pip install -q causal-conv1d>=1.4.0 --no-build-isolation
!pip install -q mamba-ssm>=2.0.0 --no-build-isolation

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")

try:
    from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
    print("\n[SUCCESS] Official fused CUDA mamba_ssm kernel is LOADED! Fast SSM Mamba = True")
except Exception as e:
    print("\n[!] Standard build failed, attempting direct compatibility installation...")
    !pip install -q torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121
    !pip install -q causal-conv1d>=1.4.0 mamba-ssm>=2.0.0 --no-build-isolation
    from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
    print("\n[SUCCESS] Fused CUDA mamba_ssm kernel successfully installed via fallback!")

### 🏗️ Step 2: WaterMamba Model Architecture Definition

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
    HAS_MAMBA = True
except Exception:
    HAS_MAMBA = False

def selective_scan_ref(u, delta, A, B, C, D, delta_bias):
    b, d, l = u.shape
    g = B.shape[1]
    u, delta = u.float(), F.softplus(delta.float() + delta_bias[None, :, None])
    B = B.float().repeat_interleave(d // g, dim=1)
    C = C.float().repeat_interleave(d // g, dim=1)
    x = u.new_zeros(b, d, A.shape[1])
    ys = []
    for i in range(l):
        dt = delta[:, :, i, None]
        x = torch.exp(dt * A) * x + dt * B[:, :, :, i] * u[:, :, i, None]
        ys.append((x * C[:, :, :, i]).sum(-1))
    return torch.stack(ys, -1) + u * D[None, :, None]

def selective_scan(u, delta, A, B, C, D, delta_bias):
    if HAS_MAMBA and u.is_cuda:
        return selective_scan_fn(u, delta, A, B, C, D, z=None,
                                 delta_bias=delta_bias, delta_softplus=True)
    return selective_scan_ref(u, delta, A, B, C, D, delta_bias)

class S6(nn.Module):
    def __init__(self, d, d_state=16, K=1):
        super().__init__()
        self.d, self.N, self.K = d, d_state, K
        self.R = math.ceil(d / 16)
        self.x_proj_weight = nn.Parameter(torch.randn(K, self.R + 2 * d_state, d) * (d ** -0.5))
        self.dt_proj_weight = nn.Parameter(torch.empty(K, d, self.R).uniform_(-self.R ** -0.5, self.R ** -0.5))
        dt = torch.exp(torch.rand(K, d) * (math.log(0.1) - math.log(1e-3)) + math.log(1e-3)).clamp(min=1e-4)
        self.dt_bias = nn.Parameter(dt + torch.log(-torch.expm1(-dt)))
        self.A_log = nn.Parameter(torch.log(torch.arange(1, d_state + 1).float()).repeat(K * d, 1))
        self.D = nn.Parameter(torch.ones(K * d))

    def forward(self, xs):
        b, K, d, L = xs.shape
        x_dbl = torch.einsum("bkdl,kcd->bkcl", xs, self.x_proj_weight)
        dts, Bs, Cs = torch.split(x_dbl, [self.R, self.N, self.N], dim=2)
        dts = torch.einsum("bkrl,kdr->bkdl", dts, self.dt_proj_weight)
        y = selective_scan(
            xs.reshape(b, K * d, L).contiguous(), dts.reshape(b, K * d, L).contiguous(),
            -torch.exp(self.A_log.float()), Bs.contiguous(), Cs.contiguous(),
            self.D.float(), self.dt_bias.reshape(-1).float())
        return y.view(b, K, d, L)

class SOSS(nn.Module):
    def __init__(self, dim, d_state=16, expand=2):
        super().__init__()
        di = dim * expand
        self.in_proj = nn.Linear(dim, di * 2)
        self.dw = nn.Conv2d(di, di, 3, padding=1, groups=di)
        self.ssm = S6(di, d_state, K=4)
        self.norm = nn.LayerNorm(di)
        self.out_proj = nn.Linear(di, dim)

    def forward(self, x):
        B, C, H, W = x.shape
        L = H * W
        u, z = self.in_proj(x.permute(0, 2, 3, 1)).chunk(2, -1)
        u = F.silu(self.dw(u.permute(0, 3, 1, 2)))
        xh = torch.stack([u.flatten(2), u.transpose(2, 3).flatten(2)], 1)
        xs = torch.cat([xh, xh.flip(-1)], 1)
        y = self.ssm(xs)
        y = (y[:, 0] + y[:, 2].flip(-1)
             + y[:, 1].reshape(B, -1, W, H).transpose(2, 3).reshape(B, -1, L)
             + y[:, 3].flip(-1).reshape(B, -1, W, H).transpose(2, 3).reshape(B, -1, L))
        y = self.norm(y.view(B, -1, H, W).permute(0, 2, 3, 1))
        y = y * F.silu(z)
        return self.out_proj(y).permute(0, 3, 1, 2)

class ChannelScan(nn.Module):
    def __init__(self, length, d_state=16):
        super().__init__()
        self.fwd = S6(length, d_state)
        self.bwd = S6(length, d_state)

    def forward(self, y):
        seq = y.transpose(1, 2).unsqueeze(1)
        f = self.fwd(seq)[:, 0]
        b = self.bwd(seq.flip(-1))[:, 0].flip(-1)
        g = F.silu(y).transpose(1, 2)
        return torch.sigmoid(((f + b) * g).transpose(1, 2))

class CCOSS(nn.Module):
    def __init__(self, dim, h, w, d_state=16):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv2d(dim, dim, 1), nn.BatchNorm2d(dim), nn.ReLU(inplace=True))
        self.scan_h = ChannelScan(h, d_state)
        self.scan_w = ChannelScan(w, d_state)
        self.dw = nn.Conv2d(dim, dim, 1, groups=dim)

    def forward(self, x):
        B, C, H, W = x.shape
        yh, yw = x.mean(3), x.mean(2)
        y = self.conv(torch.cat([yh, yw], 2).unsqueeze(2)).squeeze(2)
        yh1, yw1 = torch.split(y, [H, W], dim=2)
        ah = self.scan_h(yh1).unsqueeze(3)
        aw = self.scan_w(yw1).unsqueeze(2)
        return ah * aw * x * torch.softmax(self.dw(x), 1) + x

class MSFFN(nn.Module):
    def __init__(self, dim):
        super().__init__()
        def branch(k):
            p = k // 2
            return nn.Sequential(
                nn.Conv2d(dim, dim, k, padding=p),
                nn.Conv2d(dim, dim, k, padding=p, groups=dim),
                nn.ReLU(inplace=True),
                nn.Conv2d(dim, dim, k, padding=p),
                nn.Conv2d(dim, dim, k, padding=p, groups=dim)
            )
        self.b1, self.b3, self.b5 = branch(1), branch(3), branch(5)
        self.merge = nn.Conv2d(3 * dim, dim, 1)

    def forward(self, x):
        return self.merge(torch.cat([self.b1(x), self.b3(x), self.b5(x)], 1))

class SCOSSBlock(nn.Module):
    def __init__(self, dim, size):
        super().__init__()
        h, w = size
        self.n1 = nn.GroupNorm(1, dim)
        self.n2 = nn.GroupNorm(1, dim)
        self.soss = SOSS(dim)
        self.ccoss = CCOSS(dim, h, w)
        self.ffn = MSFFN(dim)

    def forward(self, x):
        z = x + self.ccoss(self.soss(self.n1(x)))
        return z + self.ffn(self.n2(z))

class WaterMamba(nn.Module):
    def __init__(self, dim=32, img_size=256, blocks=1):
        super().__init__()
        assert img_size % 8 == 0
        dims = [dim * (2 ** i) for i in range(4)]
        sizes = [(img_size // (2 ** i), img_size // (2 ** i)) for i in range(4)]
        stage = lambda i: nn.Sequential(*[SCOSSBlock(dims[i], sizes[i]) for _ in range(blocks)])
        self.embed = nn.Conv2d(3, dim, 3, padding=1)
        self.enc = nn.ModuleList([stage(i) for i in range(3)])
        self.down = nn.ModuleList([nn.Conv2d(dims[i], dims[i + 1], 2, 2) for i in range(3)])
        self.bottleneck = stage(3)
        self.up = nn.ModuleList([nn.ConvTranspose2d(dims[i + 1], dims[i], 2, 2) for i in range(3)])
        self.fuse = nn.ModuleList([nn.Conv2d(2 * dims[i], dims[i], 1) for i in range(3)])
        self.dec = nn.ModuleList([stage(i) for i in range(3)])
        self.out = nn.Conv2d(dim, 3, 3, padding=1)

    def forward(self, x):
        f, skips = self.embed(x), []
        for i in range(3):
            f = self.enc[i](f)
            skips.append(f)
            f = self.down[i](f)
        f = self.bottleneck(f)
        for i in reversed(range(3)):
            f = self.fuse[i](torch.cat([self.up[i](f), skips[i]], 1))
            f = self.dec[i](f)
        return self.out(f) + x

print("WaterMamba Model architecture compiled successfully!")

### 📊 Step 3: Dataset Loader and Evaluation Metrics (PSNR & SSIM)

In [ ]:
import os, glob, time
from pathlib import Path
import pandas as pd
from PIL import Image
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader

def calc_psnr(pred, gt, max_val=1.0):
    mse = torch.mean((pred - gt) ** 2)
    if mse == 0:
        return float("inf")
    return 20.0 * math.log10(max_val / math.sqrt(mse.item()))

def _gaussian_window(window_size=11, sigma=1.5, channel=3):
    coords = torch.arange(window_size, dtype=torch.float32) - window_size // 2
    g = torch.exp(-(coords ** 2) / (2 * (sigma ** 2)))
    g = g / g.sum()
    w_2d = g.unsqueeze(1) @ g.unsqueeze(0)
    return w_2d.unsqueeze(0).unsqueeze(0).repeat(channel, 1, 1, 1)

def calc_ssim(pred, gt, window_size=11, sigma=1.5):
    channel = pred.size(1)
    w = _gaussian_window(window_size, sigma, channel).to(pred.device, dtype=pred.dtype)
    mu1 = F.conv2d(pred, w, padding=window_size // 2, groups=channel)
    mu2 = F.conv2d(gt, w, padding=window_size // 2, groups=channel)
    mu1_sq, mu2_sq, mu1_mu2 = mu1.pow(2), mu2.pow(2), mu1 * mu2
    sigma1_sq = F.conv2d(pred * pred, w, padding=window_size // 2, groups=channel) - mu1_sq
    sigma2_sq = F.conv2d(gt * gt, w, padding=window_size // 2, groups=channel) - mu2_sq
    sigma12 = F.conv2d(pred * gt, w, padding=window_size // 2, groups=channel) - mu1_mu2
    c1, c2 = 0.01 ** 2, 0.03 ** 2
    ssim_map = ((2 * mu1_mu2 + c1) * (2 * sigma12 + c2)) / ((mu1_sq + mu2_sq + c1) * (sigma1_sq + sigma2_sq + c2))
    return ssim_map.mean().item()

class LSUIPairDataset(Dataset):
    def __init__(self, data_dir, split="train", splits_csv=None, img_size=256, max_samples=None):
        super().__init__()
        self.data_dir = Path(data_dir)
        self.split = split
        self.img_size = img_size

        csv_file = Path(splits_csv) if splits_csv else self.data_dir / "lsui_splits.csv"
        df = pd.read_csv(csv_file)
        split_df = df[df["split"].str.lower() == split.lower()]

        self.samples = []
        for _, row in split_df.iterrows():
            fname = str(row["filename"])
            inp_p = self.data_dir / "input" / fname
            gt_p = self.data_dir / "GT" / fname
            if inp_p.exists() and gt_p.exists():
                self.samples.append((str(inp_p), str(gt_p), fname))

        if max_samples and max_samples > 0:
            self.samples = self.samples[:max_samples]
        print(f"Loaded {len(self.samples)} pairs for split '{split}' from {self.data_dir.name}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        inp_path, gt_path, fname = self.samples[idx]
        inp_img = Image.open(inp_path).convert("RGB")
        gt_img = Image.open(gt_path).convert("RGB")
        inp_img = TF.resize(inp_img, [self.img_size, self.img_size])
        gt_img = TF.resize(gt_img, [self.img_size, self.img_size])
        inp_t = TF.to_tensor(inp_img)
        gt_t = TF.to_tensor(gt_img)
        if self.split == "train":
            if torch.rand(1).item() < 0.5:
                inp_t, gt_t = inp_t.flip(-1), gt_t.flip(-1)
            if torch.rand(1).item() < 0.5:
                inp_t, gt_t = inp_t.flip(-2), gt_t.flip(-2)
        return inp_t, gt_t, fname

### 🚀 Step 4: Run Training on Kaggle GPU ($256 \times 256$, `batch_size = 2`)
Adjust `DATA_DIR` below to where your LSUI dataset is uploaded on Kaggle (e.g., `/kaggle/input/lsui-dataset` or `/kaggle/input/lsui/LSUI`).

In [ ]:
DATA_DIR = "/kaggle/input/lsui/LSUI"

if not os.path.exists(DATA_DIR):
    found = glob.glob("/kaggle/input/**/lsui_splits.csv", recursive=True)
    if found:
        DATA_DIR = os.path.dirname(found[0])
        print(f"Auto-detected LSUI dataset at: {DATA_DIR}")
    else:
        print("[!] Please upload or link LSUI dataset to your Kaggle notebook.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 256
BATCH_SIZE = 2
EPOCHS = 50
LR = 1e-4
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

net = WaterMamba(dim=32, img_size=IMG_SIZE, blocks=1).to(DEVICE)
print(f"WaterMamba Parameters: {sum(p.numel() for p in net.parameters()) / 1e6:.2f} M")

train_loader = DataLoader(LSUIPairDataset(DATA_DIR, split="train", img_size=IMG_SIZE), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(LSUIPairDataset(DATA_DIR, split="val", img_size=IMG_SIZE), batch_size=1, shuffle=False, num_workers=2)

optimizer = torch.optim.AdamW(net.parameters(), lr=LR, betas=(0.9, 0.999), weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
criterion = nn.L1Loss()

best_psnr = -1.0
best_path = os.path.join(CHECKPOINT_DIR, "watermamba_best.pth")

print("\n--- Starting Training Loop ---")
for epoch in range(1, EPOCHS + 1):
    net.train()
    running_loss, t0 = 0.0, time.time()
    for step, (raw, ref, _) in enumerate(train_loader):
        raw, ref = raw.to(DEVICE), ref.to(DEVICE)
        optimizer.zero_grad()
        enhanced = net(raw)
        loss = criterion(enhanced, ref)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if (step + 1) % 100 == 0 or (step + 1) == len(train_loader):
            print(f"[Epoch {epoch:02d}/{EPOCHS:02d}] Step {step+1:04d}/{len(train_loader):04d} | Loss: {loss.item():.5f} | Avg: {running_loss/(step+1):.5f}")
    
    scheduler.step()
    
    net.eval()
    v_loss, v_psnr, v_ssim = 0.0, 0.0, 0.0
    with torch.no_grad():
        for raw, ref, _ in val_loader:
            raw, ref = raw.to(DEVICE), ref.to(DEVICE)
            enh = net(raw).clamp(0.0, 1.0)
            v_loss += criterion(enh, ref).item()
            v_psnr += calc_psnr(enh, ref)
            v_ssim += calc_ssim(enh, ref)
    v_loss /= len(val_loader)
    v_psnr /= len(val_loader)
    v_ssim /= len(val_loader)
    
    print(f"--> Epoch {epoch:02d} Done ({time.time()-t0:.1f}s) | Val Loss: {v_loss:.5f} | Val PSNR: {v_psnr:.2f} dB | Val SSIM: {v_ssim:.4f}")
    if v_psnr > best_psnr:
        best_psnr = v_psnr
        torch.save(net.state_dict(), best_path)
        print(f"    [★ Best Model Saved] PSNR: {best_psnr:.2f} dB")

print(f"\nTraining complete! Best checkpoint saved at: {best_path}")

### 🖼️ Step 5: Test Model & Display Visual Enhancement Results

In [ ]:
import matplotlib.pyplot as plt

net.load_state_dict(torch.load(best_path, map_location=DEVICE))
net.eval()

test_loader = DataLoader(LSUIPairDataset(DATA_DIR, split="test", img_size=IMG_SIZE), batch_size=1, shuffle=False)
print(f"Evaluating on {len(test_loader)} test images...")

test_psnr, test_ssim = 0.0, 0.0
sample_visuals = []

with torch.no_grad():
    for i, (raw, ref, fnames) in enumerate(test_loader):
        raw, ref = raw.to(DEVICE), ref.to(DEVICE)
        enh = net(raw).clamp(0.0, 1.0)
        test_psnr += calc_psnr(enh, ref)
        test_ssim += calc_ssim(enh, ref)
        
        if i < 4:
            sample_visuals.append((raw[0].cpu().permute(1, 2, 0).numpy(),
                                   enh[0].cpu().permute(1, 2, 0).numpy(),
                                   ref[0].cpu().permute(1, 2, 0).numpy(),
                                   fnames[0]))

print(f"\n" + "="*60)
print(f"Final Test Results:")
print(f"Mean PSNR: {test_psnr / len(test_loader):.2f} dB")
print(f"Mean SSIM: {test_ssim / len(test_loader):.4f}")
print("="*60)

fig, axes = plt.subplots(len(sample_visuals), 3, figsize=(12, 4 * len(sample_visuals)))
for idx, (inp, out, gt, name) in enumerate(sample_visuals):
    axes[idx, 0].imshow(inp)
    axes[idx, 0].set_title(f"Raw Input ({name})")
    axes[idx, 0].axis("off")
    
    axes[idx, 1].imshow(out)
    axes[idx, 1].set_title(f"WaterMamba Enhanced")
    axes[idx, 1].axis("off")
    
    axes[idx, 2].imshow(gt)
    axes[idx, 2].set_title(f"Ground Truth")
    axes[idx, 2].axis("off")

plt.tight_layout()
plt.show()